# SupportPilot AI — Data Preprocessing & Dataset Split

## Tujuan

Notebook ini digunakan untuk mempersiapkan dataset sebelum proses
Machine Learning.

Tahapan yang dilakukan:

- Memuat raw dataset
- Memvalidasi struktur data
- Membuat normalized instruction untuk duplicate detection
- Menghapus normalized duplicate
- Mempertahankan teks asli untuk modeling
- Menentukan feature dan target
- Membuat Train, Validation, dan Test split secara stratified
- Memvalidasi hasil split
- Menyimpan processed dataset

Preprocessing dilakukan secara minimal agar variasi bahasa alami,
typo, dan karakteristik teks customer tetap dipertahankan.

In [1]:
# import library
from pathlib import Path
import json

import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
RAW_DATA_PATH = Path(
    "../data/raw/bitext_retail_ecommerce.csv"
)

PROCESSED_DIR = Path(
    "../data/processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw data :", RAW_DATA_PATH)
print("Output   :", PROCESSED_DIR)

Raw data : ..\data\raw\bitext_retail_ecommerce.csv
Output   : ..\data\processed


In [3]:
# validasi file
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_DATA_PATH}"
    )

print("Dataset ditemukan.")

Dataset ditemukan.


In [4]:
# load raw dataset
df = pd.read_csv(
    RAW_DATA_PATH
)

print(
    f"Jumlah data awal : {len(df):,}"
)

df.head()

Jumlah data awal : 44,884


,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...


In [5]:
# validasi kolom
required_columns = {
    "instruction",
    "intent",
    "category",
    "tags",
    "response",
}

missing_columns = (
    required_columns - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Validasi kolom: BERHASIL")

Validasi kolom: BERHASIL


In [6]:
# validasi null
critical_columns = [
    "instruction",
    "intent",
]

null_summary = (
    df[critical_columns]
    .isnull()
    .sum()
)

null_summary

instruction    0
intent         0
dtype: int64

In [7]:
if null_summary.sum() > 0:
    raise ValueError(
        "Ditemukan missing value pada kolom penting."
    )

print("Tidak ditemukan missing value pada feature/target.")

Tidak ditemukan missing value pada feature/target.


In [8]:
# buat normalized instruction
df_work = df.copy()

df_work["normalized_instruction"] = (
    df_work["instruction"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [9]:
# hitung duplicate
normalized_duplicate_count = (
    df_work[
        "normalized_instruction"
    ]
    .duplicated()
    .sum()
)

print(
    f"Normalized duplicate sebelum cleaning: "
    f"{normalized_duplicate_count:,}"
)

Normalized duplicate sebelum cleaning: 57


In [10]:
# hapus normalized duplicate
before_cleaning = len(df_work)

df_clean = (
    df_work
    .drop_duplicates(
        subset="normalized_instruction",
        keep="first",
    )
    .copy()
)

after_cleaning = len(df_clean)

removed_rows = (
    before_cleaning - after_cleaning
)

print(
    f"Sebelum cleaning : {before_cleaning:,}"
)

print(
    f"Setelah cleaning : {after_cleaning:,}"
)

print(
    f"Data dihapus      : {removed_rows:,}"
)

Sebelum cleaning : 44,884
Setelah cleaning : 44,827
Data dihapus      : 57


In [11]:
# validasi duplicate sudah hilang
remaining_duplicates = (
    df_clean[
        "normalized_instruction"
    ]
    .duplicated()
    .sum()
)

print(
    f"Normalized duplicate setelah cleaning: "
    f"{remaining_duplicates}"
)

Normalized duplicate setelah cleaning: 0


In [12]:
# mendefinisikan feature dan target
FEATURE_COLUMN = "instruction"
TARGET_COLUMN = "intent"

X = df_clean[FEATURE_COLUMN]
y = df_clean[TARGET_COLUMN]

print("Feature :", FEATURE_COLUMN)
print("Target  :", TARGET_COLUMN)

print(
    f"Jumlah feature : {len(X):,}"
)

print(
    f"Jumlah class   : {y.nunique()}"
)

Feature : instruction
Target  : intent
Jumlah feature : 44,827
Jumlah class   : 46


In [13]:
# split train and temporary
RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_clean["intent"],
)

print(
    f"Train     : {len(train_df):,}"
)

print(
    f"Temporary : {len(temp_df):,}"
)

Train     : 35,861
Temporary : 8,966


In [14]:
# split validation and test
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["intent"],
)

In [15]:
# check hasil split
total_data = len(df_clean)

split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "records": [
            len(train_df),
            len(validation_df),
            len(test_df),
        ],
    }
)

split_summary["percentage"] = (
    split_summary["records"]
    / total_data
    * 100
).round(2)

split_summary

,split,records,percentage
0,train,35861,80.0
1,validation,4483,10.0
2,test,4483,10.0


In [16]:
# pastikan semua intent ada
print(
    "Jumlah intent Train      :",
    train_df["intent"].nunique(),
)

print(
    "Jumlah intent Validation :",
    validation_df["intent"].nunique(),
)

print(
    "Jumlah intent Test       :",
    test_df["intent"].nunique(),
)

Jumlah intent Train      : 46
Jumlah intent Validation : 46
Jumlah intent Test       : 46


In [17]:
# check distribusi class
class_distribution = pd.DataFrame(
    {
        "train": (
            train_df["intent"]
            .value_counts(normalize=True)
        ),
        "validation": (
            validation_df["intent"]
            .value_counts(normalize=True)
        ),
        "test": (
            test_df["intent"]
            .value_counts(normalize=True)
        ),
    }
)

class_distribution.head(10)

,train,validation,test
intent,,,
add_product,0.021277,0.021414,0.021191
availability,0.021500,0.021414,0.021637
availability_in_store,0.016843,0.016730,0.016953
availability_online,0.022085,0.022083,0.022083
cancel_order,0.022225,0.022083,0.022306
change_account,0.022030,0.022083,0.021860
change_order,0.021444,0.021414,0.021414
close_account,0.022197,0.022306,0.022083
customer_service,0.022057,0.022083,0.022083


In [18]:
# leakage check
train_keys = set(
    train_df["normalized_instruction"]
)

validation_keys = set(
    validation_df["normalized_instruction"]
)

test_keys = set(
    test_df["normalized_instruction"]
)

In [19]:
train_validation_overlap = (
    train_keys & validation_keys
)

train_test_overlap = (
    train_keys & test_keys
)

validation_test_overlap = (
    validation_keys & test_keys
)

print(
    "Train ↔ Validation overlap:",
    len(train_validation_overlap),
)

print(
    "Train ↔ Test overlap:",
    len(train_test_overlap),
)

print(
    "Validation ↔ Test overlap:",
    len(validation_test_overlap),
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0


In [20]:
# buat maping dari intent
intent_labels = sorted(
    df_clean["intent"].unique()
)

label2id = {
    label: idx
    for idx, label in enumerate(
        intent_labels
    )
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(
    f"Jumlah label: {len(label2id)}"
)

Jumlah label: 46


In [21]:
list(label2id.items())[:10]

[('add_product', 0),
 ('availability', 1),
 ('availability_in_store', 2),
 ('availability_online', 3),
 ('cancel_order', 4),
 ('change_account', 5),
 ('change_order', 6),
 ('close_account', 7),
 ('customer_service', 8),
 ('damaged_delivery', 9)]

In [22]:
columns_to_save = [
    "instruction",
    "intent",
    "category",
    "tags",
    "response",
]

In [23]:
cleaned_df_to_save = (
    df_clean[columns_to_save]
    .reset_index(drop=True)
)

train_to_save = (
    train_df[columns_to_save]
    .reset_index(drop=True)
)

validation_to_save = (
    validation_df[columns_to_save]
    .reset_index(drop=True)
)

test_to_save = (
    test_df[columns_to_save]
    .reset_index(drop=True)
)

In [24]:
# save processed dataset
cleaned_df_to_save.to_csv(
    PROCESSED_DIR / "cleaned_dataset.csv",
    index=False,
)

train_to_save.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False,
)

validation_to_save.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False,
)

test_to_save.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False,
)

In [25]:
# simpan label mapping
label_mapping = {
    "label2id": label2id,
    "id2label": {
        str(key): value
        for key, value in id2label.items()
    },
}

In [26]:
with open(
    PROCESSED_DIR / "label_mapping.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_mapping,
        file,
        indent=4,
    )

In [27]:
# verifikasi file
for file_path in sorted(
    PROCESSED_DIR.iterdir()
):
    print(file_path.name)

cleaned_dataset.csv
label_mapping.json
test.csv
train.csv
validation.csv


## Kesimpulan Preprocessing

Tahap preprocessing menghasilkan dataset yang siap digunakan
untuk proses Machine Learning.

### Cleaning

- Raw dataset tetap dipertahankan tanpa perubahan.
- Missing value tidak ditemukan.
- Exact duplicate tidak ditemukan.
- Normalized duplicate dihapus menggunakan lowercase dan
  whitespace normalization sebagai duplicate key.
- Teks asli pada kolom `instruction` tetap dipertahankan.
- Typo, bahasa informal, dan profanity tidak dihapus.

### Dataset Split

Dataset dibagi menjadi:

- 80% Training Data
- 10% Validation Data
- 10% Test Data

Pembagian dilakukan menggunakan stratified split berdasarkan
kolom `intent` agar distribusi 46 class tetap proporsional pada
setiap split.

### Data Leakage

Normalized instruction overlap diperiksa antara:

- Train dan Validation
- Train dan Test
- Validation dan Test

Tidak boleh terdapat normalized instruction yang sama pada
lebih dari satu split.

### Output

Processed dataset disimpan dalam:

- `data/processed/train.csv`
- `data/processed/validation.csv`
- `data/processed/test.csv`
- `data/processed/cleaned_dataset.csv`
- `data/processed/label_mapping.json`

Dataset siap digunakan pada tahap berikutnya yaitu Machine
Learning baseline dan model comparison.